# HAI Repeated Experiments (Colab Runner)

This notebook is designed to run the heavy pipeline in a Google Colab runtime from VS Code.

It executes:
1. run_multicollinearity_cleanup.py
2. run_repeated_experiments.py (20 seeds, checkpoint/resume enabled)

Setup behavior in Cell 2:
- Mounts Google Drive (Colab only)
- Auto-clones the public GitHub repository into /content/HAI_April_2026
- Falls back to manual path selection if needed

Run order recommendation:
- Run Cell 2 (project setup)
- Run Cell 3 (dependency install)
- Run Cell 4 (GPU/resource check and tuning setup)
- Continue with the remaining cells

Repository URL: https://github.com/ZahirAhmadChaudhry/HAI_April_2026

In [2]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/ZahirAhmadChaudhry/HAI_April_2026.git'
REPO_NAME = 'HAI_April_2026'

# Optional manual override:
# MANUAL_PROJECT_DIR = Path('/content/drive/MyDrive/HAI_April_2026')
MANUAL_PROJECT_DIR = None

clone_target = Path('/content') / REPO_NAME
if IN_COLAB and not clone_target.exists():
    print('Cloning public repository into', clone_target)
    try:
        subprocess.check_call(['git', 'clone', REPO_URL, str(clone_target)])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Git clone failed. Check network or set MANUAL_PROJECT_DIR to an existing local copy.'
        ) from exc

candidates = [
    MANUAL_PROJECT_DIR,
    clone_target,
    Path('/content/drive/MyDrive/HAI_April_2026'),
    Path('/content/HAI_April_2026'),
    Path.cwd(),
]

project_dir = None
for c in candidates:
    if c is None:
        continue
    c = Path(c)
    if (c / 'clean_hai_dataset.csv').exists() and (c / 'run_repeated_experiments.py').exists():
        project_dir = c
        break

if project_dir is None:
    raise FileNotFoundError(
        'Could not find HAI_April_2026. Set MANUAL_PROJECT_DIR to the correct path.'
    )

os.chdir(project_dir)
print('Project directory:', project_dir)
print('Python executable:', sys.executable)
print('Current working directory:', Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning public repository into /content/HAI_April_2026
Project directory: /content/HAI_April_2026
Python executable: /usr/bin/python3
Current working directory: /content/HAI_April_2026


In [3]:
import subprocess
import sys

packages = [
    'numpy',
    'pandas',
    'scikit-learn',
    'imbalanced-learn',
    'optuna',
    'xgboost',
    'lightgbm',
    'catboost',
    'shap',
    'joblib',
    'matplotlib',
    'seaborn',
    'scipy'
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('Dependencies installed.')

Dependencies installed.


In [ ]:
import os
import subprocess
import traceback

def _run(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True, check=False)
    return p.returncode, (p.stdout or "").strip(), (p.stderr or "").strip()

cpu_threads = os.cpu_count() or 2
thread_vars = [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]
for var in thread_vars:
    os.environ[var] = str(cpu_threads)
os.environ["HAI_NUM_THREADS"] = str(cpu_threads)

gpu_detected = False
gpu_info = "No GPU detected via nvidia-smi"

rc, out, err = _run(["nvidia-smi", "-L"])
if rc == 0 and out:
    gpu_detected = True
    gpu_info = out.splitlines()[0]
    print("GPU detected:", gpu_info)
    rc2, out2, err2 = _run(["nvidia-smi"])
    if out2:
        print("\n=== nvidia-smi (truncated) ===")
        print("\n".join(out2.splitlines()[:20]))
else:
    print("GPU check:", gpu_info)
    if err:
        print("nvidia-smi error:", err)

if gpu_detected:
    # Quick smoke test to verify that GPU training actually works.
    try:
        import numpy as np
        from xgboost import XGBClassifier

        rng = np.random.default_rng(42)
        X = rng.normal(size=(256, 10))
        y = (X[:, 0] + X[:, 1] > 0).astype(int)

        model = XGBClassifier(
            n_estimators=20,
            max_depth=3,
            learning_rate=0.1,
            tree_method="hist",
            device="cuda",
            eval_metric="logloss",
            n_jobs=cpu_threads,
            random_state=42,
        )
        model.fit(X, y)
        print("XGBoost GPU smoke test: PASS")
    except Exception as exc:
        gpu_detected = False
        print("XGBoost GPU smoke test: FAIL, fallback to CPU.")
        print(str(exc))
        traceback.print_exc(limit=1)

os.environ["HAI_USE_GPU"] = "1" if gpu_detected else "0"
if gpu_detected and not os.environ.get("HAI_GPU_DEVICES", "").strip():
    os.environ["HAI_GPU_DEVICES"] = "0"

print("\nResource configuration ready:")
print("HAI_USE_GPU=", os.environ["HAI_USE_GPU"])
print("HAI_NUM_THREADS=", os.environ["HAI_NUM_THREADS"])
print("Thread env vars set to", cpu_threads)
print("This configuration will be used by run_repeated_experiments.py")

In [4]:
import subprocess
import sys

print('Running multicollinearity cleanup...')
subprocess.check_call([sys.executable, 'run_multicollinearity_cleanup.py'], cwd=project_dir)
print('Part 1 complete.')

Running multicollinearity cleanup...
Part 1 complete.


In [5]:
import json
import pandas as pd
from pathlib import Path

cleaned_path = Path(project_dir) / 'cleaned_feature_sets.json'
report_path = Path(project_dir) / 'multicollinearity_cleanup_report.md'
heatmap_path = Path(project_dir) / 'figures' / 'correlation_matrix.png'

payload = json.loads(cleaned_path.read_text(encoding='utf-8'))
print('Model A cleaned feature count:', len(payload['model_a_cleaned']))
print('Model B cleaned feature count:', len(payload['model_b_cleaned']))
print('Dropped features:', list(payload['dropped_features'].keys()))
print('Heatmap exists:', heatmap_path.exists())

print('\nReport preview:')
print('-' * 80)
print(report_path.read_text(encoding='utf-8')[:2000])

Model A cleaned feature count: 20
Model B cleaned feature count: 33
Dropped features: ['nurse_staffing_count', 'nurse_aide_staffing_count', 'nurse_anesthetist_staffing_count', 'total_staffing_count']
Heatmap exists: True

Report preview:
--------------------------------------------------------------------------------
# Multicollinearity Cleanup Report

Computed on training set only (2019+2020), after imputation and before encoding.

## 1. Correlation Pairs with |r| > 0.80
| feature_1 | feature_2 | r | abs_r |
| --- | --- | --- | --- |
| nurse_anesthetist_staffing_etp | nurse_anesthetist_staffing_count | 1 | 1 |
| nurse_aide_staffing_etp | nurse_aide_staffing_count | 0.999413 | 0.999413 |
| total_staffing_etp | total_staffing_count | 0.997995 | 0.997995 |
| nurse_aide_staffing_etp | total_staffing_count | 0.985932 | 0.985932 |
| nurse_aide_staffing_count | total_staffing_count | 0.984996 | 0.984996 |
| nurse_aide_staffing_etp | total_staffing_etp | 0.981851 | 0.981851 |
| nurse_staffing

In [6]:
import subprocess
import sys

# Set to True to run the full 20-seed repeated experiment (can take hours).
RUN_FULL_REPEATED_EXPERIMENT = True

if RUN_FULL_REPEATED_EXPERIMENT:
    print('Running repeated experiments (20 seeds, checkpoint-enabled)...')
    subprocess.check_call([sys.executable, 'run_repeated_experiments.py'], cwd=project_dir)
    print('Part 2 complete.')
else:
    print('Skipped Part 2. Set RUN_FULL_REPEATED_EXPERIMENT=True when ready.')

Running repeated experiments (20 seeds, checkpoint-enabled)...


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import pandas as pd

expected_files = [
    'repeated_experiment_results.csv',
    'repeated_experiment_summary.md',
    'repeated_shap_stability.csv',
    'figures/auc_distribution_boxplot.png',
    'figures/auc_difference_histogram.png',
    'figures/shap_stability_plot.png'
]

missing = []
for rel in expected_files:
    p = Path(project_dir) / rel
    if not p.exists():
        missing.append(rel)

if missing:
    print('Missing outputs:')
    for m in missing:
        print('-', m)
else:
    print('All repeated-experiment outputs are present.')

results_path = Path(project_dir) / 'repeated_experiment_results.csv'
if results_path.exists():
    df = pd.read_csv(results_path)
    print('Results shape:', df.shape)
    print('Unique seeds:', sorted(df['seed'].unique().tolist())[:5], '...')
    print('Model rows per seed (should be 8):')
    print(df.groupby('seed').size().describe())
    display(df.head())

## Notes

- run_repeated_experiments.py uses checkpointing via repeated_experiment_checkpoint.csv.
- If runtime disconnects, re-run the repeated experiment cell (now Cell 7); completed seeds are skipped automatically.
- The repository is public, so token setup is not required for cloning.
- Cell 4 validates GPU availability and configures CPU/GPU environment variables consumed by the experiment script.
- After completion, check repeated_experiment_summary.md for paper-ready aggregate results.